# 什么时候需要微调向量模型

本 Notebook 用《南瓜书》完成一个可复现的 query→evidence 微调实验。目标不是证明“微调一定更好”，而是回答一个可检验的问题：当解析、分块和检索基线已经固定后，领域内语义训练数据能否改善同分布的检索任务？

实验固定使用 `BAAI/bge-small-zh-v1.5`。训练配置只看 dev，选定后才打开 frozen test；训练 positive 始终是 PDF 原文 evidence，不使用生成答案。


In [1]:
from pathlib import Path
import json
import sys

COURSE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if candidate.name == "C7 高级 RAG 技巧" and (candidate / "common").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from common.dataset import load_dataset
from common.training_utils import load_finetune_pairs

DATASET_ROOT = COURSE_ROOT / "data" / "dataset"
PACKAGE = load_dataset(DATASET_ROOT)
PAIRS = load_finetune_pairs(
    DATASET_ROOT / "finetune_pairs.jsonl",
    evidence_rows=PACKAGE["evidence"],
    query_rows=PACKAGE["queries"],
)

split_counts = {split: sum(row["split"] == split for row in PAIRS) for split in ("train", "dev", "test")}
page_counts = {split: len({row["page"] for row in PAIRS if row["split"] == split}) for split in split_counts}
construction = PACKAGE["manifest"]["data_construction"]
print(json.dumps({
    "model": "BAAI/bge-small-zh-v1.5",
    "pair_counts": split_counts,
    "source_page_counts": page_counts,
    "data_construction": {
        key: construction[key]
        for key in (
            "method", "generator_model", "auditor_model",
            "source_pages_reviewed", "source_pages_retained", "source_pages_excluded",
            "source_chunks_reviewed", "primary_audit_input_count",
            "primary_audit_retained_item_count", "retained_item_count",
            "retained_pair_count_by_split", "generation_schema_rejection_count",
            "rejected_item_count", "chunk_token_budget", "model_max_seq_length",
            "human_verified",
        )
    },
}, ensure_ascii=False, indent=2))


{
  "model": "BAAI/bge-small-zh-v1.5",
  "pair_counts": {
    "train": 101,
    "dev": 28,
    "test": 34
  },
  "source_page_counts": {
    "train": 55,
    "dev": 14,
    "test": 17
  },
  "data_construction": {
    "method": "pumpkin_book_fixed_token_chunk_semantic_query_evidence_v3",
    "generator_model": "glm-4-flash",
    "auditor_model": "glm-4-flash",
    "source_pages_reviewed": 100,
    "source_pages_retained": 100,
    "source_pages_excluded": 0,
    "source_chunks_reviewed": 287,
    "primary_audit_input_count": 282,
    "primary_audit_retained_item_count": 273,
    "retained_item_count": 163,
    "retained_pair_count_by_split": {
      "train": 101,
      "dev": 28,
      "test": 34
    },
    "generation_schema_rejection_count": 3,
    "rejected_item_count": 119,
    "chunk_token_budget": 384,
    "model_max_seq_length": 512,
    "human_verified": false
  }
}


## 数据为什么适合这个实验

先把 100 个 PDF 页面按 BGE tokenizer 做成与问题无关、最多 384 tokens 的固定原文 chunk；生成、审核、训练和评估始终使用同一段文字，不按问题或答案临时裁块。每个 chunk 最多生成 1 道原文足以支持的问法，不强制凑齐四种类型；连续不满足 schema 的生成会显式拒绝，语义审核失败的候选也不会入集。第二次独立 `glm-4-flash` 调用检查问题自包含、positive 可完整回答、答案事实有依据和 support quote 完整绑定。当前保留的 query→evidence pair 为 101/28/34（train/dev/frozen test）；其中 100 个页面是候选固定 chunk 语料页，实际进入这三组 pair 的页数为 55/14/17。

这里的 101 条 train 足够演示并真实检验一个小规模领域微调闭环，但不等于生产场景的数据已经充分。是否继续扩充，应看错误是否集中在未覆盖主题、问法或文档类型，而不是套用一个固定的“至少多少条”。问法由 `glm-4-flash` 生成和初审，并经独立模型复核剔除明确错误；全部记录均为 `human_verified=false`，所以结论只覆盖本组合成问法分布。


In [2]:
sample = next(row for row in PAIRS if row["split"] == "train")
print(json.dumps({
    "query": sample["query"],
    "positive_evidence_id": sample["positive_evidence_id"],
    "page": sample["page"],
    "review_type": sample["review_type"],
    "semantic_review": sample["semantic_review"],
    "positive_prefix": sample["positive"][:180],
}, ensure_ascii=False, indent=2))


{
  "query": "本书的撰写初衷是什么？",
  "positive_evidence_id": "evi_ft_chunk_769e19c3c0fb037e",
  "page": 14,
  "review_type": "model_assisted_semantic_training_v3",
  "semantic_review": {
    "question_self_contained": true,
    "positive_fully_supports": true,
    "answer_fully_supported": true,
    "support_quotes_sufficient": true,
    "status": "passed",
    "review_type": "model_assisted_semantic_training_v3",
    "human_verified": false,
    "note": "通过真实 glm-4-flash 语义审核；未声称人工核验。"
  },
  "positive_prefix": "第1 章 绪论 本章作为“西瓜书”的开篇，主要讲解什么是机器学习以及机器学习的相关数学符号，为后续内容作 铺垫，并未涉及复杂的算法理论，因此阅读本章时只需耐心梳理清楚所有概念和数学符号即可。此外，在 阅读本章前建议先阅读西瓜书目录前页的《主要符号表》，它能解答在阅读“西瓜书”过程中产生的大部 分对数学符号的疑惑。 本章也作为本书的开篇，笔者在此赘述一下本书的"
}


## 监督数据结构与损失选择

先确认一条样本表达的监督关系，再选择损失函数。下面五种监督结构的输入和目标不同，不能只因为都包含句子就互换。

| 数据结构 | 输入 | 训练目标 | 对应损失 | 假负例风险与适用条件 |
|---|---|---|---|---|
| query-positive | `(query, positive_evidence)`；MNRL 以 batch 中相同位置的配对为正例 | 提高 query 与正确 evidence 的相似度，并把同批其他 evidence 排在后面 | `MultipleNegativesRankingLoss`（MNRL）、`MegaBatchMarginLoss` | 其他 evidence 只有在确实不相关时才能当负例；同一 query 的多个相关 evidence、重复片段或同主题片段进入同批会形成假负例。适合已有可靠正例、尚未整理显式负例的检索训练 |
| 带分数句对 | `(sentence_A, sentence_B, label/score)`；`ContrastiveLoss` 使用 0/1，`CosineSimilarityLoss` 使用连续相似度分数 | 二分类时拉近正例、推远负例；连续分数时让余弦相似度拟合给定分数 | `ContrastiveLoss`、`CosineSimilarityLoss` | 把“未标注”误写成 0 会把潜在正例当成负例；只有相似关系定义稳定、标签含义明确时才适合 |
| 带类别的句对 | `(sentence_A, sentence_B, class_label)` | 在两个句向量及其差值等组合表示上训练分类头 | `SoftmaxLoss` | class 是句对关系类别，不等于相似度连续分数；适合 NLI 等句对分类式监督，训练目标与直接优化检索排序不同 |
| triplet | `(anchor, positive, negative)` | 让正例距离小于负例距离，并至少保留 margin 间隔 | `TripletLoss` | negative 必须经过判断且确实不应匹配；随机或跨主题的简单负例信息少，误选的 hard negative 反而会放大错误。适合有明确难负例的排序任务 |
| 带标签样本 | `(sentence, class_label)` 单句与类别标签；批内按类别构造正负关系 | 同类样本靠近、不同类样本分开 | `BatchAllTripletLoss`、`BatchHardTripletLoss`、`BatchHardSoftMarginTripletLoss`、`BatchSemiHardTripletLoss` | 标签表示类别而不是某个 query 的证据关系；类别边界不稳定或每类样本太少时，不宜直接当检索监督 |

### 损失函数的输入语义

- `ContrastiveLoss` 的标签不能凭印象反转。按本页依赖的 Sentence Transformers 5.2.0 实现，`label=1` 表示相似/正例，损失中的 `distance²` 会把两者拉近；`label=0` 表示不相似/负例，`max(margin-distance, 0)²` 会把距离推到至少 `margin`。因此只有经过判断的 0/1 关系才适合这个损失，未标注不等于负例。
- `CosineSimilarityLoss` 接收连续的 `score`，目标是让两个句向量的余弦相似度接近该分数；它不是把 0/1 标签自动解释成检索正负的替代写法。分数尺度和标注规则必须在训练、验证中保持一致。
- `TripletLoss` 接收三段文字，要求 `d(anchor, positive) + margin <= d(anchor, negative)`；违反这个间隔时才产生损失。negative 的语义必须是“对这个 anchor 不相关或不应排在 positive 前”，不能只因为没有被标注就当负例。
- `MultipleNegativesRankingLoss` 接收 query-positive 句对，在一个 batch 的相似度矩阵中把第 `i` 个 query 与第 `i` 个 evidence 配成正例，并把其他列当作负例。因此批次越大不必然越好：批内假负例会直接提供相反梯度，应按完整 qrels 做分组或改用显式负例损失。
- `MegaBatchMarginLoss` 也使用 anchor-positive 句对，但会在较大的 batch 中为每个 anchor 寻找最难的其他 positive 作为负例，再按 margin 训练。它能扩大难负例范围，也会放大假负例与内存成本，因此需要先检查 qrels 和重复/同义 evidence。
- `SoftmaxLoss` 在两个句向量的组合表示上增加分类头，用句对 class label 做交叉熵训练；它适合句对分类或 NLI 风格监督，不应把分类准确率直接写成检索 Recall 的提升。
- Batch Triplet 系列先用类别标签在批内构造三元组：`BatchAllTripletLoss` 使用有效三元组，`BatchHardTripletLoss` 选择最难正例与负例，`BatchSemiHardTripletLoss` 选择符合半难条件的负例。`BatchHardSoftMarginTripletLoss` 使用 soft margin，不要求固定 margin。它们都需要批内有同类正例和异类负例，不能只给每类一个样本。

参数更新的抽象写法是 `theta <- theta - lr*grad`，其中 `grad` 是损失对参数的梯度；优化器可以额外使用动量或权重衰减，但基本的梯度方向仍是减去梯度，而不是加上梯度。

本页当前的 163 条 query→evidence 数据按 query-positive 组织（101/28/34），并配合 qrels 处理 MNRL 的批内假负例，因此只支撑下面这次 MNRL 实验；没有用这份数据运行或比较 `MegaBatchMarginLoss`、`ContrastiveLoss`、`CosineSimilarityLoss`、`SoftmaxLoss`、`TripletLoss` 或 Batch Triplet 系列。损失函数的数据格式选择可继续对照 [Sentence Transformers Loss Overview](https://www.sbert.net/docs/sentence_transformer/loss_overview.html)。


## 实验：训练与配置选择

训练使用 `MultipleNegativesRankingLoss`。batch sampler 同时检查 positive ID 与完整 qrels：共享任一已知正例的 query 不会进入同一 batch，避免相关证据被当成假负例。

候选学习率为 `5e-6 / 1e-5 / 2e-5`，均训练 1 epoch、batch size 16、seed 42。本页保存的输出由下方代码直接调用 `run_experiment` 生成，每次在临时目录保存候选模型；选择顺序为 dev Recall@10、Recall@5、Recall@3、Recall@1、MRR，选定模型后才同时评估原始 BGE 与微调模型的 frozen test。模型缓存必须预先存在，依赖、模型或数据缺失会直接报错，不使用 fallback。

运行前按[教程首页](../README.md#运行准备)进入 C7 根目录，安装 `requirements-c7.txt` 并选择 `llm-universe-c7` kernel；BGE 缓存准备命令见[本章 README](README.md#运行实验)。


In [3]:
from common.training_utils import run_experiment
import tempfile

CACHE_ROOT = COURSE_ROOT / ".cache"
CACHE_ROOT.mkdir(exist_ok=True)
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="c2_embedding_finetune_", dir=CACHE_ROOT))
RESULT = run_experiment(
    PACKAGE,
    PAIRS,
    OUTPUT_DIR,
    epochs=1,
    batch_size=16,
    learning_rate=2e-5,
    seed=42,
    device="cpu",
)
print(json.dumps({
    "selected_candidate_id": RESULT["selected_candidate_id"],
    "selected_config": RESULT["selection_candidates"][RESULT["selected_candidate_id"] - 1]["config"],
    "frozen_test_opened_after_selection": RESULT["frozen_test_opened_after_selection"],
}, ensure_ascii=False, indent=2))


/usr/local/Caskroom/miniconda/base/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
  "selected_candidate_id": 3,
  "selected_config": {
    "epochs": 1,
    "learning_rate": 2e-05
  },
  "frozen_test_opened_after_selection": true
}


In [4]:
def metric_view(metrics):
    return {key: round(metrics[key], 4) for key in ("Recall@1", "Recall@3", "Recall@5", "Recall@10", "mrr")}

candidate_dev = []
candidate_training = []
for candidate in RESULT["selection_candidates"]:
    row = {
        "candidate_id": candidate["candidate_id"],
        "epochs": candidate["config"]["epochs"],
        "learning_rate": candidate["config"]["learning_rate"],
    }
    row.update(metric_view(candidate["dev"]))
    candidate_dev.append(row)
    candidate_training.append({
        "candidate_id": candidate["candidate_id"],
        "epochs": candidate["training"]["epochs"],
        "batch_size": candidate["training"]["batch_size"],
        "learning_rate": candidate["training"]["learning_rate"],
        "optimizer_steps": candidate["training"]["optimizer_steps"],
        "loss_first": candidate["training"]["loss_first"],
        "loss_last": candidate["training"]["loss_last"],
    })

selected_candidate = next(
    candidate
    for candidate in RESULT["selection_candidates"]
    if candidate["candidate_id"] == RESULT["selected_candidate_id"]
)
training = selected_candidate["training"]
selected_training = {
    key: training[key]
    for key in (
        "epochs", "batch_size", "learning_rate", "train_pair_count",
        "optimizer_steps", "loss_first", "loss_last",
        "unique_positive_per_batch", "train_unique_positive_count",
        "batch_count_per_epoch",
    )
}
comparison = RESULT["comparison"]["frozen_test"]
REPORT = {
    "report_schema": "c2-compact-report-v2",
    "dataset": {
        "version": PACKAGE["manifest"]["dataset_version"],
        "pair_count_by_split": {
            split: sum(row["split"] == split for row in PAIRS)
            for split in ("train", "dev", "test")
        },
        "dev_query_count": RESULT["dev_query_count"],
        "frozen_test_query_count": RESULT["frozen_test_query_count"],
    },
    "candidate_dev": candidate_dev,
    "candidate_training": candidate_training,
    "selected_candidate_id": RESULT["selected_candidate_id"],
    "selected_model_dir": RESULT["selected_model_dir"],
    "selected_training": selected_training,
    "frozen_test": {
        "baseline": metric_view(RESULT["baseline"]["frozen_test"]),
        "finetuned": metric_view(RESULT["finetuned"]["frozen_test"]),
        "rank_change_counts": {
            "improved": comparison["improved_count"],
            "degraded": comparison["degraded_count"],
            "unchanged": comparison["unchanged_count"],
        },
        "degraded_query_ids": comparison["degraded_query_ids"],
        "per_query_rank": [
            {
                "query_id": row["query_id"],
                "baseline_rank": row["baseline_rank"],
                "finetuned_rank": row["finetuned_rank"],
            }
            for row in comparison["per_query"]
        ],
    },
}
print(json.dumps(REPORT, ensure_ascii=False, indent=2))


{
  "report_schema": "c2-compact-report-v2",
  "dataset": {
    "version": "2026-09-11",
    "pair_count_by_split": {
      "train": 101,
      "dev": 28,
      "test": 34
    },
    "dev_query_count": 28,
    "frozen_test_query_count": 34
  },
  "candidate_dev": [
    {
      "candidate_id": 1,
      "epochs": 1,
      "learning_rate": 5e-06,
      "Recall@1": 0.7857,
      "Recall@3": 0.9286,
      "Recall@5": 0.9643,
      "Recall@10": 0.9643,
      "mrr": 0.8634
    },
    {
      "candidate_id": 2,
      "epochs": 1,
      "learning_rate": 1e-05,
      "Recall@1": 0.7857,
      "Recall@3": 0.9643,
      "Recall@5": 0.9643,
      "Recall@10": 0.9643,
      "mrr": 0.8782
    },
    {
      "candidate_id": 3,
      "epochs": 1,
      "learning_rate": 2e-05,
      "Recall@1": 0.8214,
      "Recall@3": 0.9643,
      "Recall@5": 0.9643,
      "Recall@10": 1.0,
      "mrr": 0.8968
    }
  ],
  "candidate_training": [
    {
      "candidate_id": 1,
      "epochs": 1,
      "batch_size": 1

## 结果怎么读

本次冻结测试包含 34 条 query：Recall@10 从 0.9412 提升到 1.0000，Recall@5 从 0.8235 提升到 0.9706，Recall@3 保持 0.7941，Recall@1 从 0.5882 提升到 0.6471，MRR 从 0.6987 提升到 0.7591。34 个问题中 9 个排名改善、3 个退化、22 个不变。
因此，这次实验只支持：在 `glm-4-flash` 生成和初审、并经独立模型复核的南瓜书合成问法分布上，微调模型的整体检索指标提高；它不证明自然用户问法上的稳定收益，也不表示每个问题都改善。这是一个小规模教程闭环，不是生产充分数据；3 个退化问题的身份和 baseline/finetuned rank 已持久化在上一个代码单元的 compact report 中，生产使用前还需独立收集真实读者问题验证。
